# Intake Orchestrator — Multi-Agent Workflow with GraphBuilder

The **Intake Orchestrator** is the supervisor agent in the claims processing pipeline. Unlike the specialist agents built in the previous notebooks, the Intake Orchestrator coordinates the work of other agents across the full claim lifecycle.

In this notebook we build the **Phase 1 (Pre-Processing)** scope of the supervisor. In later notebooks, the same supervisor will be extended to handle Claims Processing (Adjudicator, human-in-the-loop) and Post-Processing (Communicator Agent).

### Agent Classification

The Intake Orchestrator is a **Supervisor Agent** — it owns the claim lifecycle and delegates work to specialist agents:

| | Specialist Agent | Supervisor Agent |
|---|---|---|
| **Role** | Performs a single specialist task | Coordinates multiple agents across phases |
| **Control flow** | LLM decides which tool to call | Supervisor decides which agent/tool to invoke |
| **Scope** | One step in the pipeline | Full claim lifecycle (pre-processing → processing → post-processing) |
| **State** | Single agent message history | Maintains claim context across all phases |
| **Examples** | Authenticator, Extractor, Policy Verification | Intake Orchestrator (this notebook) |

### What's New in This Notebook

| Concept | Description |
|---|---|
| **Supervisor Agent** | Explicit agent with system prompt, model, and `@tool` functions |
| **Strands `GraphBuilder`** | Pre-processing sub-graph wired as a tool the supervisor calls |
| **Shared agent modules** | Importing specialist agents as reusable components |
| **S3 document retrieval** | `@tool` for fetching claim documents from S3 |
| **DynamoDB claim state** | `@tool` for writing claim metadata |
| **Multi-agent orchestration** | Supervisor delegates to Authenticator, Extractor, and Policy Verification via graph |

### How It Fits in the Pipeline

```
                     ┌──────────────────────────┐
                     │   Supervisor Agent       │
                     │   (Intake Orchestrator)  │
                     └────────────┬─────────────┘
                                  │
           ┌──────────────────────┼──────────────────────┐
           ▼                      ▼                      ▼
 ┌─────────────────┐   ┌─────────────────┐   ┌─────────────────┐
 │  Authenticator  │   │    Extractor    │   │    Policy       │
 │     Agent       │   │     Agent       │   │  Verification   │
 └─────────────────┘   └─────────────────┘   └─────────────────┘
```

The supervisor retrieves documents from S3, runs the pre-processing graph (authenticate → extract → verify), and persists the claim state to DynamoDB. In later notebooks, it will also coordinate the Adjudicator and Communicator agents.

## Setup

Note we import shared agent modules from `insurance-claims-processing/agents/` which exposes `build_agent()` factory functions that return fresh agent instances. This factory pattern ensures each graph node gets a clean agent (empty message history) per invocation.

In [ ]:
import boto3
import json
import sys
import os
import re
from pathlib import Path
from datetime import datetime, timezone
from decimal import Decimal

from strands import Agent, tool
from strands.models import BedrockModel
from strands.multiagent import GraphBuilder
from strands.tools.mcp import MCPClient
from mcp import stdio_client, StdioServerParameters

# ── Path setup ──────────────────────────────────────────────────────────────────
# Walk up from CWD until we find the agents/ directory (works regardless of
# whether the kernel starts in notebooks/ or notebooks/03_multi_agent_orchestration/)
_cwd = Path.cwd()
REPO_ROOT = _cwd
for _parent in [_cwd] + list(_cwd.parents):
    if (_parent / "agents").is_dir() and (_parent / "mcp_servers").is_dir():
        REPO_ROOT = _parent
        break
AGENTS_DIR = REPO_ROOT / "agents"
MCP_SERVER_PATH = REPO_ROOT / "mcp_servers" / "socotra_mock"
MOCK_DATA_PATH = MCP_SERVER_PATH / "mock_data.json"

# Add repo root to Python path so we can import shared modules
sys.path.insert(0, str(REPO_ROOT))

# ── Import shared agent modules ──────────────────────────────────────────────────
from agents import authenticator_agent
from agents import extractor_agent
from agents import policy_verification_agent

# ── AWS configuration ────────────────────────────────────────────────────────────
REGION = "us-east-1"
S3_BUCKET = os.environ.get("WORKSHOP_S3_BUCKET", "")
if not S3_BUCKET:
    # Auto-detect: find the workshop bucket by naming convention
    for _b in boto3.client("s3", region_name=REGION).list_buckets().get("Buckets", []):
        if _b["Name"].startswith("agentic-workshop-"):
            S3_BUCKET = _b["Name"]
            break
assert S3_BUCKET, "Could not find workshop S3 bucket. Set WORKSHOP_S3_BUCKET env var or ensure the CloudFormation stack is deployed."
S3_PREFIX = "claims-processing/claimant-data/"
DYNAMODB_TABLE = "claims-metadata"

# ── AWS clients ─────────────────────────────────────────────────────────────────
s3_client = boto3.client("s3", region_name=REGION)
dynamodb = boto3.resource("dynamodb", region_name=REGION)

print("✅ Setup complete")
print(f"   REPO_ROOT       : {REPO_ROOT}")
print(f"   Agents dir      : {AGENTS_DIR}")
print(f"   S3 bucket       : s3://{S3_BUCKET}/{S3_PREFIX}")
print(f"   DynamoDB table  : {DYNAMODB_TABLE}")
print(f"   MCP server      : {MCP_SERVER_PATH}")

## Part A — Define the Supervisor's Tools

The Intake Orchestrator is a Strands `Agent` with three `@tool` functions:

1. **`retrieve_claim_documents`** — Downloads claim PDFs from S3 to a local working directory
2. **`run_preprocessing_graph`** — Executes the 3-node GraphBuilder sub-graph (authenticate → extract → verify)
3. **`persist_claim_to_dynamodb`** — Writes the final claim record to DynamoDB

Each tool encapsulates a distinct phase of the pre-processing workflow. The supervisor decides when to call each tool based on its system prompt and the claim context.

### Why Tools, Not Loose Code?

In the previous iteration, S3 retrieval and DynamoDB persistence were standalone code cells. But the supervisor agent needs to own the full claim lifecycle — including document retrieval and persistence. By making these `@tool` functions, the supervisor can reason about when and whether to call them. This also makes the supervisor extensible: in later notebooks, we add tools for adjudication and communication without changing the core agent.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Tool 1: Retrieve claim documents from S3
# ─────────────────────────────────────────────────────────────────────────────

LOCAL_DOCS_DIR = Path("claim_documents")
LOCAL_DOCS_DIR.mkdir(exist_ok=True)

@tool
def retrieve_claim_documents(s3_bucket: str, s3_prefix: str) -> str:
    """Download claim PDF documents from S3 to a local working directory.

    Args:
        s3_bucket: S3 bucket name containing claim documents
        s3_prefix: S3 key prefix for the claim's documents
    """
    response = s3_client.list_objects_v2(Bucket=s3_bucket, Prefix=s3_prefix)
    s3_objects = response.get("Contents", [])

    downloaded = []
    for obj in s3_objects:
        key = obj["Key"]
        filename = key.split("/")[-1]
        if not filename or not filename.endswith(".pdf"):
            continue
        local_path = LOCAL_DOCS_DIR / filename
        s3_client.download_file(s3_bucket, key, str(local_path))
        downloaded.append(str(local_path))
        print(f"   ✅ {filename} ({obj['Size']:,} bytes)")

    return json.dumps({
        "status": "success",
        "documents_retrieved": len(downloaded),
        "document_paths": downloaded,
    })


print("✅ retrieve_claim_documents tool defined")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Tool 2: Persist claim record to DynamoDB
# ─────────────────────────────────────────────────────────────────────────────

@tool
def persist_claim_to_dynamodb(
    claim_id: str,
    policy_number: str,
    claimant_name: str,
    date_of_death: str,
    auth_summary: str,
    verification_summary: str,
) -> str:
    """Write the final claim record to DynamoDB after pre-processing completes.

    Args:
        claim_id: Unique claim identifier (e.g. CLM-2026-00101)
        policy_number: Policy number (e.g. WL-4582-1093)
        claimant_name: Name of the claimant
        date_of_death: Date of death (YYYY-MM-DD)
        auth_summary: Summary of authentication result
        verification_summary: Summary of policy verification result
    """
    from botocore.exceptions import ClientError

    json_match = re.search(
        r'\{[\s\S]*verification_decision[\s\S]*\}',
        verification_summary
    )
    if json_match:
        try:
            verification_decision = json.loads(json_match.group())
        except json.JSONDecodeError:
            verification_decision = {"raw_response": verification_summary[:3000]}
    else:
        verification_decision = {"raw_response": verification_summary[:3000]}

    def float_to_decimal(obj):
        if isinstance(obj, float):
            return Decimal(str(obj))
        elif isinstance(obj, dict):
            return {k: float_to_decimal(v) for k, v in obj.items()}
        elif isinstance(obj, list):
            return [float_to_decimal(i) for i in obj]
        return obj

    claim_record = {
        "claim_id": claim_id,
        "policy_number": policy_number,
        "claimant_name": claimant_name,
        "date_of_death": date_of_death,
        "auth_result_summary": auth_summary[:2000],
        "verification_decision": verification_decision,
        "created_at": datetime.now(timezone.utc).isoformat() + "Z",
        "stage": "intake_orchestration_complete",
        "pipeline": "intake_orchestrator_v1",
    }

    try:
        table = dynamodb.create_table(
            TableName=DYNAMODB_TABLE,
            KeySchema=[{"AttributeName": "claim_id", "KeyType": "HASH"}],
            AttributeDefinitions=[{"AttributeName": "claim_id", "AttributeType": "S"}],
            BillingMode="PAY_PER_REQUEST",
        )
        table.wait_until_exists()
        print(f"   Created DynamoDB table: {DYNAMODB_TABLE}")
    except ClientError as e:
        if e.response["Error"]["Code"] == "ResourceInUseException":
            table = dynamodb.Table(DYNAMODB_TABLE)
        else:
            raise

    claim_record_ddb = float_to_decimal(claim_record)
    table.put_item(Item=claim_record_ddb)

    return json.dumps({
        "status": "success",
        "table": DYNAMODB_TABLE,
        "claim_id": claim_id,
        "stage": "intake_orchestration_complete",
    })


print("✅ persist_claim_to_dynamodb tool defined")

## Part B — Build the Pre-Processing Graph

### Why GraphBuilder for Claims Processing?

Strands offers several orchestration patterns — plain workflow (manual Python function chaining), Swarm (peer-to-peer handoffs), and GraphBuilder (directed graph with typed nodes and edges). We use **GraphBuilder** here because the nature of insurance claims processing demands it:

1. **Strict ordering with regulatory consequences.** A claim must be authenticated before documents are extracted, and extraction must complete before policy verification begins. Processing documents for an unverified claimant wastes compute and creates compliance risk. GraphBuilder enforces this sequence declaratively through edges — the framework guarantees node B never runs before node A completes.

2. **Conditional routing based on upstream outcomes.** If authentication fails (fraudulent claimant, lapsed policy, identity mismatch), the pipeline must stop immediately — there is no reason to extract documents or verify policy terms for a rejected claim. GraphBuilder supports this natively with conditional edges: `add_edge("authenticate", "extract", condition=check_auth_passed)`. A plain workflow would require manual if/else branching that becomes brittle as the pipeline grows.

3. **Typed execution traces for auditability.** Insurance claims are auditable events. GraphBuilder returns a `GraphResult` with per-node outputs, timing, and the exact path taken through the graph. This trace answers questions like: *Which agent produced this verification decision? How long did authentication take? Did the graph short-circuit?* A plain workflow gives you none of this without building it yourself.

4. **Extensibility for later phases.** In later notebooks we  will add an Adjudicator node with human-in-the-loop routing and a Communicator node. GraphBuilder lets us add nodes and edges incrementally without rewriting the orchestration logic. The graph is a living blueprint of the claims pipeline.

In short: claims processing is a **governed, sequential pipeline with conditional branching and audit requirements** — exactly the problem GraphBuilder is designed to solve.

### Graph Structure

```
authenticate ──┬── (passed) ──→ extract ──→ verify_policy
               └── (failed) ──→ [graph ends early]
```

### MCP Client Lifecycle

The Socotra MCP client is shared by the Authenticator and Policy Verification agents. We start it once before the supervisor runs and keep it open for the full pipeline.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Configure the Socotra MCP client — shared by Authenticator and Policy
# Verification agents. Started once, kept open for the full pipeline.
# ─────────────────────────────────────────────────────────────────────────────

SOCOTRA_SERVER_SCRIPT = str(MCP_SERVER_PATH / "server.py")

socotra_mcp_client = MCPClient(
    lambda: stdio_client(
        StdioServerParameters(
            command=sys.executable,
            args=[SOCOTRA_SERVER_SCRIPT],
            env={
                "MOCK_DATA_PATH": str(MOCK_DATA_PATH),
                **os.environ,
            }
        )
    )
)

# ─────────────────────────────────────────────────────────────────────────────
# Tool 3: Run the pre-processing graph
#
# The graph is built inside the tool so MCP tools (which require an active
# connection) are available at execution time.
# ─────────────────────────────────────────────────────────────────────────────

# Module-level flag for conditional routing
auth_passed = True

@tool
def run_preprocessing_graph(claim_prompt: str) -> str:
    """Run the 3-node pre-processing graph: authenticate, extract, verify_policy.

    The graph uses GraphBuilder to wire the specialist agents in sequence.
    If authentication fails, the graph ends early (no extraction or verification).

    Args:
        claim_prompt: Full claim submission prompt with all claim details
    """
    global auth_passed
    auth_passed = True

    mcp_tools = socotra_mcp_client.list_tools_sync()
    print(f"   🔧 MCP tools: {[t.tool_name for t in mcp_tools]}")

    # Build specialist agents
    auth_agent = authenticator_agent.build_agent(mcp_tools)

    output_dir = Path("extracted_output")
    output_dir.mkdir(exist_ok=True)
    extract_agent = extractor_agent.build_agent(output_dir=output_dir)

    verify_agent = policy_verification_agent.build_agent(mcp_tools)

    # Conditional routing
    def check_auth_passed(state):
        return auth_passed

    # Wire the graph
    builder = GraphBuilder()
    builder.add_node(auth_agent, "authenticate")
    builder.add_node(extract_agent, "extract")
    builder.add_node(verify_agent, "verify_policy")
    builder.set_entry_point("authenticate")
    builder.add_edge("authenticate", "extract", condition=check_auth_passed)
    builder.add_edge("extract", "verify_policy")

    graph = builder.build()
    result = graph(claim_prompt)

    return str(result)


print("✅ Socotra MCP client configured")
print(f"   Server: {SOCOTRA_SERVER_SCRIPT}")
print()
print("✅ run_preprocessing_graph tool defined")
print("   Graph: authenticate ──┬── (passed) ──→ extract ──→ verify_policy")
print("                        └── (failed) ──→ [graph ends]")

## Part C — Define the Supervisor Agent

Now we define the Intake Orchestrator as an explicit Strands `Agent`. The supervisor has:

- A **system prompt** describing its Phase 1 scope and the tools available to it
- A **model** (Nova 2 Lite — lightweight, since specialist agents do the heavy reasoning)
- Three **`@tool` functions**: `retrieve_claim_documents`, `run_preprocessing_graph`, `persist_claim_to_dynamodb`

The supervisor reasons about the claim submission, decides to retrieve documents, run the pre-processing graph, and persist the results. In later notebooks, additional tools (adjudication, communication) will be added to the same agent.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Define the Intake Orchestrator (Supervisor Agent)
# ─────────────────────────────────────────────────────────────────────────────

SUPERVISOR_SYSTEM_PROMPT = """
You are the Intake Orchestrator — the supervisor agent for an Insurance Claims Processing worfklow.

Your Role:
You coordinate the full claim lifecycle. In this phase (Phase 1: Pre-Processing), you:
1. Retrieve claim documents from S3
2. Run the pre-processing graph (authenticate → extract → verify_policy)
3. Persist the claim record to DynamoDB

Your Tools:
- **retrieve_claim_documents(s3_bucket, s3_prefix)**: Download claim PDFs from S3
- **run_preprocessing_graph(claim_prompt)**: Execute the 3-node specialist agent graph
- **persist_claim_to_dynamodb(...)**: Write the final claim record to DynamoDB

##Workflow
When you receive a claim submission:
1. Call retrieve_claim_documents to download the claim documents from S3
2. Build a detailed prompt with all claim details and the EXACT document_paths returned by retrieve_claim_documents. Do NOT construct filenames yourself.
3. Call run_preprocessing_graph with that prompt
4. Review the graph output for authentication status and verification decision
5. Call persist_claim_to_dynamodb with the claim details and results
6. Summarize the outcome

## Guidelines
- Always retrieve documents before running the graph
- Always persist results, even if authentication failed — we need a record of rejected claims
- Include the claim_id, policy_number, claimant_name, and date_of_death when persisting
- For auth_summary, use the authentication portion of the graph output
- For verification_summary, use the policy verification portion of the graph output
- In your final summary, report what YOU completed (documents retrieved, graph executed, record persisted) 
— do NOT echo the specialist agents' next-step recommendations. The pre-processing phase is complete when you persist the record. The next phase (Adjudication) is handled separately.
"""

supervisor_model = BedrockModel(
    model_id="us.amazon.nova-2-lite-v1:0",
    region_name=REGION,
    additional_request_fields={
        "reasoningConfig": {"type": "enabled", "maxReasoningEffort": "low"}
    },
)

intake_orchestrator = Agent(
    model=supervisor_model,
    system_prompt=SUPERVISOR_SYSTEM_PROMPT,
    tools=[
        retrieve_claim_documents,
        run_preprocessing_graph,
        persist_claim_to_dynamodb,
    ],
)

print("✅ Intake Orchestrator (Supervisor Agent) defined")
print(f"   Model : us.amazon.nova-2-lite-v1:0")
print(f"   Tools : retrieve_claim_documents, run_preprocessing_graph, persist_claim_to_dynamodb")

## Part D — Run the Full Pipeline

Now we invoke the supervisor with a claim submission. The MCP server is started once and kept open for the entire pipeline run — both the Authenticator and Policy Verification agents share the same MCP connection.

The supervisor will:
1. Call `retrieve_claim_documents` to download PDFs from S3
2. Call `run_preprocessing_graph` to run authenticate → extract → verify
3. Call `persist_claim_to_dynamodb` to write the claim record
4. Summarize the outcome

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Run the Intake Orchestrator
# ─────────────────────────────────────────────────────────────────────────────

CLAIM_SUBMISSION = {
    "claim_id": "CLM-2026-00101",
    "policy_number": "WL-4582-1093",
    "claimant_name": "Lisa Doe",
    "date_of_death": "2026-01-15",
    "death_circumstances": "Natural causes — congestive heart failure at residence",
    "documents_submitted": [
        "death_certificate", "policy_document", "medical_records",
        "will_document", "trust_document", "beneficiary_id", "police_report"
    ],
    "documents_required": [
        "death_certificate", "claim_form", "beneficiary_id", "policy_document"
    ],
}

claim = CLAIM_SUBMISSION
docs_submitted = ", ".join(claim["documents_submitted"])
docs_required = ", ".join(claim["documents_required"])

supervisor_prompt = (
    "## New Claim Submission\n\n"
    f"**Claim ID:** {claim['claim_id']}\n"
    f"**Policy Number:** {claim['policy_number']}\n"
    f"**Claimant Name:** {claim['claimant_name']}\n"
    f"**Date of Death:** {claim['date_of_death']}\n"
    f"**Death Circumstances:** {claim['death_circumstances']}\n\n"
    f"### Documents Submitted\n{docs_submitted}\n\n"
    f"### Required Documents\n{docs_required}\n\n"
    f"### S3 Location\n"
    f"Bucket: {S3_BUCKET}\n"
    f"Prefix: {S3_PREFIX}\n\n"
    "Please process this claim through the full pre-processing pipeline: "
    "retrieve documents, run the pre-processing graph, and persist the results."
)

print("🚀 Starting Intake Orchestrator")
print("=" * 60)
print(f"   Claim ID : {claim['claim_id']}")
print(f"   Policy   : {claim['policy_number']}")
print(f"   Claimant : {claim['claimant_name']}")
print("=" * 60)

with socotra_mcp_client:
    result = intake_orchestrator(supervisor_prompt)

print()
print("=" * 60)
print("✅ Intake Orchestrator Pipeline Complete")
print("=" * 60)

## Part E — Inspect the Pipeline Results

Let’s examine what the supervisor produced. The result contains the supervisor’s full response, including tool call results from each phase of the pipeline.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Inspect the pipeline results
# ─────────────────────────────────────────────────────────────────────────────

print("📊 Pipeline Results Summary")
print("=" * 60)
print()

final_response = str(result)
print("Supervisor response (first 2000 chars):")
print(final_response[:2000])
print()

# Check extracted files
output_dir = Path("extracted_output")
if output_dir.exists():
    extracted_files = list(output_dir.glob("*_extracted.json"))
    print(f"📄 Extracted JSON files: {len(extracted_files)}")
    for f in extracted_files:
        print(f"   ✅ {f.name}")

## Part F — Read Back from DynamoDB

Confirm the claim record was persisted correctly by reading it back from DynamoDB.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Read back the claim record from DynamoDB
# ─────────────────────────────────────────────────────────────────────────────

table = dynamodb.Table(DYNAMODB_TABLE)
claim_id = CLAIM_SUBMISSION["claim_id"]

response_ddb = table.get_item(Key={"claim_id": claim_id})
item = response_ddb.get("Item", {})

if item:
    print(f"✅ Claim record retrieved from DynamoDB")
    print(f"   Claim ID  : {item['claim_id']}")
    print(f"   Policy    : {item.get('policy_number', 'N/A')}")
    print(f"   Claimant  : {item.get('claimant_name', 'N/A')}")
    print(f"   Stage     : {item.get('stage', 'N/A')}")
    print(f"   Created   : {item.get('created_at', 'N/A')}")
    print(f"   Pipeline  : {item.get('pipeline', 'N/A')}")
    print()
    print("   Verification decision (first 500 chars):")
    vd = item.get('verification_decision', {})
    print(f"   {json.dumps(vd, indent=2, default=str)[:500]}")
else:
    print(f"❌ Claim record not found for {claim_id}")

## ✅ Notebook Complete: Intake Orchestrator

### What You Built

- **Intake Orchestrator** — an explicit Supervisor Agent with system prompt, model, and `@tool` functions
- Three tools: `retrieve_claim_documents`, `run_preprocessing_graph`, `persist_claim_to_dynamodb`
- Pre-processing sub-graph using `GraphBuilder`: authenticate → extract → verify_policy
- Conditional routing: authentication failure ends the graph early
- S3 document retrieval and DynamoDB persistence as supervisor-owned tools
- Shared agent modules imported from `agents/` — reusable across notebooks

### Key Patterns Used

| Pattern | Description |
|---|---|
| **Supervisor Agent** | Explicit agent that owns the claim lifecycle. Tools give it capabilities; system prompt gives it judgment. Extensible — add tools for adjudication and communication in later notebooks. |
| **GraphBuilder sub-graph** | Pre-processing pipeline wired as a tool. The supervisor decides when to invoke it; the graph handles the specialist agent sequencing. |
| **Factory pattern for agents** | `build_agent()` returns a fresh instance per graph invocation — no state bleed between pipeline runs. |
| **Shared MCP connection** | Single MCP server process shared across Authenticator and Policy Verification agents. |
| **Conditional routing** | Authentication failure ends the graph early — no wasted compute on doomed claims. |

### What’s Next

| Notebook | Phase | What Gets Added |
|---|---|---|
| Post-Processing | Communicator Agent for claim decision notifications (next step for happy-path claims) |
| Claims Processing | Adjudicator agent, human-in-the-loop workflow (next step when human escaltion is needed |


The same Intake Orchestrator will be extended with additional tools to handle these phases.

## Cleanup (Optional)

Remove the local directories created during the pipeline run.

In [ ]:
import shutil

for folder in ['claim_documents', 'extracted_output']:
    if Path(folder).exists():
        shutil.rmtree(folder)
        print(f'✅ Removed {folder}/')
    else:
        print(f'ℹ️  {folder}/ not found (already clean)')